# Stage 12 prereq — 224×224 hand-crop video cache for VideoMAE+LoRA

Walks the same 6 paper-split sources as the landmark-cache kernel (`eng_{train,val,test}_{lex,nonlex}`) and writes a per-clip `.npz` layout under `/kaggle/working/handcrop_cache/`.

Per clip:
1. Read jpeg frames from the source directory.
2. Run MediaPipe HandLandmarker per frame.
3. Compute the **union** hand-bounding box across all valid detections, ×1.3 padding, made square, clamped.
4. Uniform-sample T=16 frames, crop with the union bbox, resize to 224×224 uint8.
5. Save as `<SIGNER>__<clip_id>.npz` with key `video` shape `[16, 224, 224, 3]`.

**Disk**: 16·224·224·3 ≈ 2.4 MB per clip uint8.  ~10 314 clips ⇒ ~25 GB total.  `np.savez` (uncompressed) so training-time load is fast.

**Wall-clock**: ~0.5-1 s per clip × 10K clips ÷ 4 workers ≈ **2-3 h CPU**.  GPU off (MediaPipe is CPU-bound).

**Resume**: per-source `_DONE` markers + per-clip overwrite check, so the kernel can be re-run after a session disconnect and will skip what is already on disk.

## After this kernel commits

Save Version → Save & Run All.  Then `Datasets → New Dataset → Notebook Output`, name it `wita-handcrop-cache-videomae`.  Attach that dataset alongside `wita-full-english-landmark-cache` in the Stage 12 training kernel.

## Cell 1 — Install + clone (MediaPipe pinned)

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk scipy opencv-python-headless --quiet
!pip install 'mediapipe==0.10.14' --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')
import mediapipe; print(f'mediapipe version: {mediapipe.__version__}')

## Cell 2 — Locate the 6 paper-split source dirs

Same hardcoded shape as the landmark-cache kernel: `<DATASET_ROOT>/eng_{split}_{subset}` with `gt.txt` files at any inner depth.  We don't extract zips here — Kaggle has already done that on mount.

In [ ]:
import os, json

# ============================================================================
# HARDCODED path — same shape the Stage 11 landmark-cache kernel uses.
# Skipping the recursive glob is essential: /kaggle/input/ contains the
# already-extracted ~10K clip subdirs (hundreds of thousands of files),
# and glob.glob('/kaggle/input/**/...', recursive=True) walks every one.
# If the dataset slug ever changes, edit DATASET_ROOT below.
# ============================================================================
DATASET_ROOT = '/kaggle/input/datasets/gaurs86/wita-full-english-122signers'

if not os.path.isdir(os.path.join(DATASET_ROOT, 'eng_train_lex')):
    # Shallow fallback: look one or two levels down under /kaggle/input/
    # for any directory that contains eng_train_lex.  No deep walk.
    cand_root = None
    for c in os.listdir('/kaggle/input'):
        p1 = os.path.join('/kaggle/input', c)
        if not os.path.isdir(p1):
            continue
        if os.path.isdir(os.path.join(p1, 'eng_train_lex')):
            cand_root = p1; break
        for c2 in os.listdir(p1):
            p2 = os.path.join(p1, c2)
            if os.path.isdir(p2) and os.path.isdir(os.path.join(p2, 'eng_train_lex')):
                cand_root = p2; break
        if cand_root: break
    assert cand_root, (
        f'eng_train_lex not under {DATASET_ROOT} or shallow fallback.\n'
        f'Attach the dataset or edit DATASET_ROOT in this cell.'
    )
    DATASET_ROOT = cand_root
print(f'DATASET_ROOT = {DATASET_ROOT}\n')

# Build the 6 expected paths directly — no recursive walk, no gt.txt count.
PATH_TO_LABEL = {}
missing = []
for split in ('train', 'val', 'test'):
    for subset in ('lex', 'nonlex'):
        outer = os.path.join(DATASET_ROOT, f'eng_{split}_{subset}')
        if os.path.isdir(outer):
            PATH_TO_LABEL[outer] = (split, subset)
            print(f'  {os.path.basename(outer):<25s} -> {split}/{subset}  [OK]')
        else:
            missing.append((split, subset))
            print(f'  eng_{split}_{subset:<7s}  -> MISSING')
assert not missing, f'missing dirs: {missing}'
print(f'\nFound 6 sources (no recursive glob; constant-time).')

## Cell 3 — Output paths + resume markers

In [ ]:
OUT_ROOT   = '/kaggle/working/handcrop_cache'
MARKER_DIR = os.path.join(OUT_ROOT, '_markers')
os.makedirs(OUT_ROOT,   exist_ok=True)
os.makedirs(MARKER_DIR, exist_ok=True)

def marker_path(p):
    split, subset = PATH_TO_LABEL[p]
    return os.path.join(MARKER_DIR, f'{split}_{subset}_DONE')

SPLIT_ORDER = {'val': 0, 'test': 1, 'train': 2}
source_paths = sorted(
    PATH_TO_LABEL.keys(),
    key=lambda p: (SPLIT_ORDER[PATH_TO_LABEL[p][0]], PATH_TO_LABEL[p][1]),
)
for p in source_paths:
    s, ss = PATH_TO_LABEL[p]
    mk    = marker_path(p)
    print(f'  {os.path.basename(p):<25s} -> {s}/{ss}  '
          f'(marker {"exists" if os.path.exists(mk) else "missing"})')

## Cell 4 — Extract per-clip handcrop videos  (~30 min per source on Kaggle CPU)

4-worker `mp.Pool` (fork-based, each worker lazy-initialises its own MediaPipe extractor).  Resume-aware: each source has a `_DONE` marker, and per-clip files are skipped if already on disk.

In [ ]:
from wita_v2.datasets.handcrop_cache import extract_dir_handcrops_parallel

T         = 16
CROP_SIZE = 224
PAD       = 1.3
N_WORKERS = 4

all_stats = {}
for p in source_paths:
    split, subset = PATH_TO_LABEL[p]
    mk            = marker_path(p)
    if os.path.exists(mk):
        print(f'[skip] {split}/{subset} already done')
        continue
    print(f'\n>>> extracting {split}/{subset}  from {p}')
    stats = extract_dir_handcrops_parallel(
        dir_path=p, out_dir=OUT_ROOT, split=split, subset=subset,
        n_workers=N_WORKERS, T=T, crop_size=CROP_SIZE, pad_factor=PAD,
        overwrite=False,
    )
    all_stats[f'{split}_{subset}'] = stats
    with open(mk, 'w') as f:
        json.dump(stats, f, indent=2, default=str)
print('\nAll sources processed.')

## Cell 5 — Sanity check: shape + dtype + counts

In [ ]:
import numpy as np, random
from pathlib import Path

all_npz = list(Path(OUT_ROOT).rglob('*.npz'))
print(f'Total .npz files: {len(all_npz)}')
assert len(all_npz) > 0, 'No clips extracted'

random.seed(42)
sample = random.sample(all_npz, min(50, len(all_npz)))
vids = []
drates = []
for p in sample:
    with np.load(p, allow_pickle=False) as d:
        v = d['video']
        vids.append(v)
        drates.append(float(d['detected']))
arr = np.stack(vids)
print(f'video shape per clip   : {arr.shape[1:]}')
print(f'video dtype            : {arr.dtype}')
print(f'value min/max          : {arr.min()} / {arr.max()}')
assert arr.shape[1:] == (T, CROP_SIZE, CROP_SIZE, 3), f'bad shape: {arr.shape[1:]}'
assert arr.dtype == np.uint8
print(f'MediaPipe detect rate  : mean={np.mean(drates):.3f}  min={np.min(drates):.3f}')
if np.mean(drates) < 0.5:
    print('WARNING: low detection rate; many clips fell back to centred-square bbox.')

print()
for split in ('train', 'val', 'test'):
    for subset in ('lex', 'nonlex'):
        n = len(list((Path(OUT_ROOT) / split / subset).glob('*.npz')))
        print(f'  {split}/{subset:<7s}: {n}')

## Cell 6 — Commit kernel + next step

1. **Save Version → Save & Run All**.  The committed kernel's output dataset contains `handcrop_cache/` (~25 GB).
2. After it commits, **Datasets → New Dataset → Notebook Output**, name it `wita-handcrop-cache-videomae`.
3. Attach **both** datasets in the Stage 12 training kernel:
   - `wita-handcrop-cache-videomae` (this one)
   - `wita-full-english-landmark-cache` (the existing one used by Stage 11)